In [0]:
%pip install azure-eventhub
dbutils.library.restartPython()

In [0]:
import json
import time
import requests
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from azure.eventhub import EventHubProducerClient, EventData

**SCOPE & PARAMETERS**

In [0]:
#Secret scope and eventhub
dbutils.widgets.text("secret_scope", "weather_scope")
secret_scope = dbutils.widgets.get("secret_scope")

dbutils.widgets.text("evh_name", "trezio2005_evh")
evh_name = dbutils.widgets.get("evh_name")
conn_string = dbutils.secrets.get(scope=secret_scope, key="evh-conn-str")

#Notebook parameters
dbutils.widgets.text("max_events_per_city", "50")
max_events = int(dbutils.widgets.get("max_events_per_city"))

cities_data = {
    "Kraków": {"lat": 50.0647, "lon": 19.9450},
    "Warszawa": {"lat": 52.2297, "lon": 21.0122},
    "Wrocław": {"lat": 51.1079, "lon": 17.0385},
    "Poznań": {"lat": 52.4064, "lon": 16.9252},
    "Gdańsk": {"lat": 54.3520, "lon": 18.6466},
    "Łódź": {"lat": 51.7592, "lon": 19.4560},
    "Szczecin": {"lat": 53.4285, "lon": 14.5528}
}
city_names = list(cities_data.keys())

dbutils.widgets.multiselect("cities", "Kraków", city_names)

selected_cities_str = dbutils.widgets.get("cities")
selected_cities = [city.strip() for city in selected_cities_str.split(",")] if selected_cities_str else []

dbutils.widgets.text("events_delay", "2", "delay in seconds between events")
events_delay = int(dbutils.widgets.get("events_delay"))

**Producer code with schema evolution scenario:**

After some events we add columns with air quailty features

In [0]:
def run_producer(city_name, lat, lon, max_events=50, delay=1):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=conn_string,
        eventhub_name=evh_name
    )

    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    air_quality_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=european_aqi,pm10,pm2_5"

    with producer:
        for i in range(max_events):
            response = requests.get(weather_url).json()
            current_weather = response.get("current_weather", {})

            payload = {
                "city": city_name,
                "temperature": current_weather.get("temperature", None),
                "wind_speed": current_weather.get("windspeed", None),
                "wind_direction": current_weather.get("winddirection", None),
                "event_timestamp": datetime.now().isoformat()
            }

            #A schema evolution scenario: after half of events we add another column with air quailty features
            if i >= max_events/2:
                response = requests.get(air_quality_url).json()
                air_quality = response.get("current", {})
                payload["air_quality"] = air_quality.get("european_aqi", None)
                payload["pm10"] = air_quality.get("pm10", None)
                payload["pm2_5"] = air_quality.get("pm2_5", None)
        
            

            event_data_batch = producer.create_batch()
            event_data_batch.add(EventData(json.dumps(payload)))
            producer.send_batch(event_data_batch)
            time.sleep(delay)

In [0]:
if selected_cities:
    with ThreadPoolExecutor(max_workers=len(selected_cities)) as executor:
        for city_name in selected_cities:
            city_loc = cities_data[city_name]

            executor.submit(
                run_producer, 
                city_name, 
                city_loc["lat"], 
                city_loc["lon"], 
                max_events=max_events, 
                delay=events_delay
            )
